# RAG Chatbot with BioBERT and LLaMA 2

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using biomedical embeddings (`BioBERT`) and a local language model (`LLaMA 2`) to answer domain-specific medical questions. The pipeline includes:

- Document loading and splitting  
- Embedding generation with `BioBERT`  
- Vector store creation using `Chroma`  
- Retrieval-based question answering with `LLaMA 2`  

This setup is ideal for medical QA tasks where factual grounding and domain relevance are critical.  
The model runs locally via `transformers` and Hugging Face pipelines, without requiring an API key.

---

In [1]:
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00


In [2]:
import os
import glob

from huggingface_hub import login

from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain.schema import Document

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

In [ ]:
import os
import glob
import shutil

from huggingface_hub import login

from langchain.schema import Document
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

### Project Directory Connection


In [3]:
!git clone https://github.com/a20190202/PLN_Medical_Flashcard.git

Cloning into 'PLN_Medical_Flashcard'...
remote: Enumerating objects: 277, done.
remote: Counting objects: 100% (277/277), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 277 (delta 93), reused 248 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (277/277), 38.25 MiB | 8.59 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (75/75), done.


In [4]:
!ls

PLN_Medical_Flashcard  sample_data


In [5]:
%cd PLN_Medical_Flashcard/pln_model

/content/PLN_Medical_Flashcard/pln_model


In [6]:
!pwd

/content/PLN_Medical_Flashcard/pln_model


# Vectorstore Generation
---

In [7]:
def read_txt_files(folder_path):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
    )

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        "source": filename,
                        "chunk_id": i,
                        "total_chunks": len(chunks)
                    }
                )
                all_docs.append(doc)

    return all_docs

all_documents = read_txt_files("data/textbooks")

## Embeddings model
### `pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`

This SentenceTransformer model is a fine-tuned version of BioBERT on multiple natural language inference (NLI) and semantic similarity datasets, including:

- MNLI, SNLI, SciNLI, SciTail, MedNLI, and STS-B

It is specifically optimized for semantic similarity tasks in the biomedical domain and is suitable for generating high-quality dense vector embeddings of medical questions, terms, or documents.

- **Base model:** `dmis-lab/biobert-base-cased-v1.1`  
- **Embedding dimensions:** `768`  
- **Use case:** Biomedical sentence embeddings for retrieval and clustering

Model link: [https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb](https://huggingface.co/pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb)


In [8]:
EMBEDDINGS = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"

In [9]:
embeddings_model = HuggingFaceEmbeddings(
        model_name=EMBEDDINGS,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/412 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Create Vector Store (Run Only Once)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings_model,
    persist_directory=f"./{EMBEDDINGS.replace('/','_')}"
)

### Save the vector store after creation

In [ ]:
# Compress the vector store directory into a ZIP file
vectorstore_dir = f"./{EMBEDDINGS.replace('/','_')}"
zip_path = f"{vectorstore_dir}.zip"
shutil.make_archive(vectorstore_dir, 'zip', vectorstore_dir)

print(f"Vector store saved and zipped as: {zip_path}")

### Load the saved vector store

In [ ]:
# Unzip the saved vector store
vectorstore_dir = f"./{EMBEDDINGS.replace('/','_')}"
zip_path = f"{vectorstore_dir}.zip"

if not os.path.exists(vectorstore_dir):
    shutil.unpack_archive(zip_path, vectorstore_dir)
    print(f"Unzipped vector store to: {vectorstore_dir}")
else:
    print(f"Directory {vectorstore_dir} already exists.")

# Load the vector store
vectorstore = Chroma(
    persist_directory=vectorstore_dir,
    embedding_function=embeddings_model
)

# RAG
---

In [ ]:
# OLlama download
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Launch the Ollama server locally
!ollama serve > /dev/null 2>&1 &
!sleep 10

In [ ]:
!ollama pull llama2:latest
!pip install -U langchain-ollama

In [ ]:
from langchain_ollama import OllamaLLM

In [ ]:
llm = OllamaLLM(
    model="llama2:latest",
    temperature=0.05,
    system=""
)

## Personalized prompt:

In [ ]:
prompt = ChatPromptTemplate.from_template(
"""
You are a biomedical assistant specialized in generating flashcards from clinical context.

Given the context below, generate exactly 5 flashcards about the following disease:
{input}

Each flashcard must cover a different topic among the following:
1. Definition or general concept
2. Etiology (causes)
3. Clinical signs and symptoms
4. Diagnosis
5. Treatment

Each flashcard should contain:
- A clear and focused question
- A medically accurate and concise answer

Use exactly the following format:

Flashcard 1:
Q: [Question]
A: [Answer]

Flashcard 2:
Q: [Question]
A: [Answer]

Flashcard 3:
Q: [Question]
A: [Answer]

Flashcard 4:
Q: [Question]
A: [Answer]

Flashcard 5:
Q: [Question]
A: [Answer]

<context>
{context}
</context>
"""
 )

## RAG Pipeline:

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Inference Test:

### __Diabetes:__

In [ ]:
question = "Diabetes"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Asthma:__

In [ ]:
question = "Asthma"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Cardiac Arrest:__

In [ ]:
question = "Cardiac Arrest"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Gastritis:__

In [ ]:
question = "Gastritis"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")

### __Stroke:__

In [ ]:
question = "Stroke"

result = retrieval_chain.invoke({
    "input": question
})


print("Respuesta:", result["answer"])
print("\nChunks recuperados:")
for doc in result["context"]:
    print(f"\n- {doc.page_content[:100]}...")